In [0]:
from pyspark.sql import functions as F

from delta.tables import DeltaTable

# ---------------- CONFIG ----------------

BRONZE_PENALTY = "workspace.slainte_bronze.penalty_rules_bronze"

GOLD_DB        = "workspace.slainte_gold"

DIM_PENALTY    = f"{GOLD_DB}.dim_penalty"

spark.sql(f"CREATE DATABASE IF NOT EXISTS {GOLD_DB}")

# ---------------- BUILD SOURCE ----------------

df_dim_penalty = (

    spark.table(BRONZE_PENALTY)

    .select(

        F.col("user_id").alias("source_user_id"),

        F.col("project_key").alias("project"),

        F.col("priority_group"),

        F.col("priority_group_name"),

        F.element_at("priority_group_labels", 1).alias("raw_label"),

        F.col("penalty_code"),

        F.col("penalty_type"),

        F.col("penalty_value"),

        F.col("penalty_unit")

    )

    .filter(

        F.col("source_user_id").isNotNull() &

        F.col("penalty_code").isNotNull()

    )

    .distinct()

)

# ---------------- PRIORITY LABEL ----------------

df_dim_penalty = (

    df_dim_penalty

    .withColumn("cleaned_label", F.regexp_replace(F.col("raw_label"), "[^0-9]", ""))

    .withColumn(

        "priority_group_label",

        F.coalesce(

            F.when(F.length(F.col("cleaned_label")) > 0,

                   F.col("cleaned_label").cast("int")),

            F.when(F.lower(F.col("priority_group_name")) == "highest", 1)

             .when(F.lower(F.col("priority_group_name")) == "high", 2)

             .when(F.lower(F.col("priority_group_name")) == "medium", 3)

             .when(F.lower(F.col("priority_group_name")) == "low", 4)

             .when(F.lower(F.col("priority_group_name")) == "lowest", 5),

            F.lit(99)

        )

    )

    .drop("cleaned_label", "raw_label")

)

# ---------------- DEDUP KEY ----------------

# business key for merge

merge_condition = """

t.source_user_id = s.source_user_id AND

t.project = s.project AND

t.penalty_code = s.penalty_code AND

t.priority_group_label = s.priority_group_label

"""

# ---------------- INITIAL LOAD (IF NOT EXISTS) ----------------

if not spark.catalog.tableExists(DIM_PENALTY):

    df_dim_penalty = df_dim_penalty.withColumn("penalty_id", F.monotonically_increasing_id() + 1)

    df_dim_penalty.write.format("delta") \
        .mode("overwrite") \
        .saveAsTable(DIM_PENALTY)

    print("✅ dim_penalty created (initial load)")

else:

    delta_table = DeltaTable.forName(spark, DIM_PENALTY)

    # ---------------- MERGE ----------------

    (

        delta_table.alias("t")

        .merge(

            df_dim_penalty.alias("s"),

            merge_condition

        )

        .whenMatchedUpdate(set={

            "priority_group": "s.priority_group",

            "priority_group_name": "s.priority_group_name",

            "penalty_type": "s.penalty_type",

            "penalty_value": "s.penalty_value",

            "penalty_unit": "s.penalty_unit"

        })

        .whenNotMatchedInsert(values={

            "penalty_id": "monotonically_increasing_id()",

            "source_user_id": "s.source_user_id",

            "project": "s.project",

            "priority_group": "s.priority_group",

            "priority_group_name": "s.priority_group_name",

            "priority_group_label": "s.priority_group_label",

            "penalty_code": "s.penalty_code",

            "penalty_type": "s.penalty_type",

            "penalty_value": "s.penalty_value",

            "penalty_unit": "s.penalty_unit"

        })

        .execute()

    )

    print("✅ dim_penalty merged successfully")

# ---------------- DISPLAY ----------------

display(

    spark.table(DIM_PENALTY)

    .orderBy("source_user_id", "project", "priority_group_label")

)
 